I used this model to test out building and training the model. I used colab's GPUs. 

In [ ]:
%pip install transformers h5py pandas pyarrow --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel, WavLMModel, AutoProcessor

from ConvStyleDataset import AudioHDF5Dataset, collate_pad

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
H5_PATH     = "path/to/audio.h5"
META_PATH   = "path/to/metadata.parquet"
BATCH_SIZE  = 8
SAMPLE_RATE = 16_000

dataset = AudioHDF5Dataset(
    h5_path=H5_PATH,
    meta_path=META_PATH,
    meta_columns=["transcription"],
    sample_rate=SAMPLE_RATE,
    # no max_len_sec -- collate_pad handles variable lengths
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pad,
    num_workers=0,  # keep at 0 until HDF5 fork-safety is confirmed
)

In [ ]:
# data batch test
batch = next(iter(loader))

audio   = batch["audio"]          # (B, T_max) float32, zero-padded
lengths = batch["lengths"]        # (B,) actual frame counts
texts   = batch["transcription"]  # list[str]

print("audio shape  :", audio.shape)
print("audio dtype  :", audio.dtype)
print("lengths      :", lengths)
print("min/max len  :", lengths.min().item(), "/", lengths.max().item())
print("transcription sample:", texts[0])
print()

# verify padding is actually zero beyond each utterance's real length
for i in range(len(lengths)):
    L = lengths[i].item()
    assert audio[i, L:].abs().max() == 0, f"item {i}: non-zero values past length {L}"

print("Padding check passed")